# Lab type: review
# Course: ML302 — Transformer Models & Fine-Tuning
# Lesson: Self-Attention and Multi-Head Attention
# Task: The code below is correct and working. Read each section, run it, then answer the judgment questions in the markdown cells below each block.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)
print('PyTorch version:', torch.__version__)

## Part 1: Scaled Dot-Product Attention

The function below implements full scaled dot-product attention. Run it and examine the outputs.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

# Small synthetic example: batch=1, seq_len=5, d_k=8
batch, seq_len, d_k = 1, 5, 8
Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)

# Without scaling
scores_unscaled = torch.matmul(Q, K.transpose(-2, -1))
print('Unscaled scores — max abs:', scores_unscaled.abs().max().item())

# With scaling
scores_scaled = scores_unscaled / math.sqrt(d_k)
print('Scaled scores   — max abs:', scores_scaled.abs().max().item())

output, weights = scaled_dot_product_attention(Q, K, V)
print('Attention weights (row 0):', weights[0, 0].detach().numpy().round(3))
print('Attention output shape:   ', output.shape)

**Question 1:** The scaling factor is `1 / sqrt(d_k)`. In this example `d_k = 8`. Try re-running the cell with `d_k = 512`. How does the max absolute value of the unscaled scores change? Why does this push the softmax into a bad gradient region?

*(Write your answer here.)*

**Question 2:** Attention weights in row 0 should sum to 1.0 (they are a probability distribution). Without the mask argument, every token can attend to every other token. In what scenario would you want to restrict which positions are visible? Give one concrete example for encoder models and one for decoder models.

*(Write your answer here.)*

## Part 2: Multiplicative vs Additive Masking

Two implementations of masked attention are shown below. One is correct; one is not.

In [ ]:
# Multiplicative masking (common AI-generated pattern)
def multiplicative_masked_attention(Q, K, V, padding_mask):
    """padding_mask: 1 = real token, 0 = padding"""
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    # Multiply by 0 for padding positions
    scores = scores * padding_mask.unsqueeze(1).unsqueeze(1).float()
    weights = F.softmax(scores, dim=-1)
    return weights

# Additive masking (correct)
def additive_masked_attention(Q, K, V, padding_mask):
    """padding_mask: 1 = real token, 0 = padding"""
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    # Fill padding positions with -inf before softmax
    scores = scores.masked_fill(padding_mask.unsqueeze(1).unsqueeze(1) == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights

# Sequence: 3 real tokens, 1 padding token
batch, seq_len, d_k = 1, 4, 8
Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)
padding_mask = torch.tensor([[1, 1, 1, 0]])  # position 3 is padding

w_mult = multiplicative_masked_attention(Q, K, V, padding_mask)
w_add  = additive_masked_attention(Q, K, V, padding_mask)

print('Multiplicative — weight on padding token (pos 3), query 0:', w_mult[0, 0, :, 3].item())
print('Additive       — weight on padding token (pos 3), query 0:', w_add[0, 0, :, 3].item())
print()
print('Multiplicative weights row 0:', w_mult[0, 0, 0].detach().numpy().round(4))
print('Additive       weights row 0:', w_add[0, 0, 0].detach().numpy().round(4))

**Question 3:** The multiplicative version sets the score for padding tokens to 0 before softmax. Why does `softmax([..., 0, ...])` not produce zero weight for that position? Show why this is mathematically wrong using the softmax formula `exp(x_i) / sum(exp(x_j))`.

*(Write your answer here.)*

**Question 4:** A production batch has sequences of length 3 and 12. The shorter sequence is padded to length 12. You use multiplicative masking. The 9 padding positions each receive a small nonzero attention weight. Describe one concrete way this leaks information from the padding into the representation of real tokens.

*(Write your answer here.)*

## Part 3: Multi-Head Attention

The implementation below is correct. Run it and examine the shapes at each step.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model must be divisible by num_heads'
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, C = x.size()
        Q = self.W_q(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        # Q, K, V: (B, num_heads, T, d_k)
        attn_out, weights = scaled_dot_product_attention(Q, K, V, mask)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, C)
        return self.W_o(attn_out), weights

d_model, num_heads = 512, 8
mha = MultiHeadAttention(d_model, num_heads)

x = torch.randn(2, 10, d_model)  # batch=2, seq_len=10
out, weights = mha(x)

print('d_k per head:', mha.d_k)                      # 64
print('Input shape: ', x.shape)                       # (2, 10, 512)
print('Output shape:', out.shape)                     # (2, 10, 512)
print('Weight shape:', weights.shape)                 # (2, 8, 10, 10)
print('Total MHA params:', sum(p.numel() for p in mha.parameters()))

**Question 5:** `d_model=512` and `num_heads=8` means each head operates in a `d_k=64` space. Compared to a single-head attention that uses `d_k=512`, what does the smaller per-head dimension mean for the attention score magnitudes before scaling? Does this change the argument for scaling by `1/sqrt(d_k)`?

*(Write your answer here.)*

**Question 6:** A colleague proposes using `num_heads=1` instead of `num_heads=8` to save compute, arguing that a single head with `d_k=512` has higher capacity than any single head in the 8-head version. Evaluate this argument: what does multi-head attention give you that a single large head cannot?

*(Write your answer here.)*

## Summary

> **For each concept, write one sentence on why it matters in practice.**

1. Scaling by `1/sqrt(d_k)`: 
2. Additive vs multiplicative masking: 
3. Multi-head vs single-head attention: 